# Data Science Project — Phase 1
## Notebook 01: Data loading & exploratory analysis

Goal: Load the Kaggle JSON files (documents, train queries, ground truth) and compute basic statistics to understand the dataset characteristics that may affect retrieval performance.

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import sys
import matplotlib.pyplot as plt

In [ ]:
cwd = Path.cwd()
ROOT_DIR = cwd if (cwd / "src").exists() else cwd.parent
DATA_DIR = ROOT_DIR / "data" / "raw"
DOCS_PATH = DATA_DIR / "docs.json"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
CACHE_DIR = ROOT_DIR / "data" / "cache"
RUNS_DIR = ROOT_DIR / "outputs" / "runs"
SUBMISSIONS_DIR = ROOT_DIR / "outputs" / "submissions"
PLOTS_PATH = ROOT_DIR / "outputs" / "eda_plots.png"

TRAIN_QUERIES_PATH = DATA_DIR / "queries_train.json"
TEST_QUERIES_PATH = DATA_DIR / "queries_test.json"
GTS_PATH = DATA_DIR / "qgts_train.json"

print("cwd:", cwd)
print("ROOT_DIR:", ROOT_DIR)
print("DATA_DIR:", DATA_DIR)


In [ ]:
def load_json(path: Path):
    """
    Load a JSON file and return the corresponding Python object.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

documents = load_json(DOCS_PATH)
train_queries = load_json(TRAIN_QUERIES_PATH)
test_queries = load_json(TEST_QUERIES_PATH)
gts = load_json(GTS_PATH)

print("Datasets loaded successfully:")
print(f"Number of documents: {len(documents):,}")
print(f"Number of train queries: {len(train_queries):,}")
print(f"Number of test queries: {len(test_queries):,}")
print(f"Number of ground truth entries: {len(gts):,}")

In [ ]:
# Inspect one example document and one example train query
print("Example document:")
print(documents[0])

print("\nExample train query:")
print(train_queries[0])

# Print a meaningful example of ground truth.
print("\nExample ground-truth entry (query_id -> relevant_doc_ids):")

if isinstance(gts, dict):           # Verification if gts is a dict.
    first_qid = next(iter(gts))     # Take the first id of the query in gts.
    rel = gts[first_qid]            # Take the documents associates.
    print(first_qid, "->", rel[:10] if isinstance(rel, list) else rel) # Print the 10 first documents.

elif isinstance(gts, list): # Verification if gts is a list.
    print("Ground truth is a list. Example element:")
    print(gts[0])

else:
    print(f"Unexpected ground truth format: {type(gts)}")
    print(gts)

## Preprocessing Method

The `content` field of each document/query is built by `add_content_field()`
in `src/data/preprocess.py`. It results from two steps: **merge** then **optional cleaning**.

---

### 1. Field Merge — `build_content()`

Source fields are concatenated in the following order, separated by a space:

```
content = title + " " + text + " " + tags
```

| Order | Field  | Expected Type        | Handling if absent/invalid          |
|-------|--------|----------------------|-------------------------------------|
| 1     | `title`| `str` or `None`      | ignored if empty after `to_string()` |
| 2     | `text` | `str` or `None`      | ignored if empty after `to_string()` |
| 3     | `tags` | `list[str]` or `str` | joined by space; ignored if empty   |

**Missing Field Handling — `to_string(v)`**

Each field goes through `to_string()` before being added:
- `None` → `""` (ignored)
- Unicode replacement character `\ufffd` (U+FFFD) → `" "` (encoding sanitization)
- Empty result after `.strip()` → ignored (not added to `content`)

Tags in list form are filtered individually (`None` and empty excluded),
then joined by a single space before concatenation.

---

### 2. Optional Cleaning — `clean_text()` (`clean=True`)

Enabled only if `add_content_field(..., clean=True)`.
The notebook statistics use `clean=False` to stay on raw text.

| Step | Transformation                                   | Example                    |
|------|--------------------------------------------------|----------------------------|
| 1    | Lowercase                                        | `"LaTeX"` → `"latex"`      |
| 2    | Multiple spaces / `\t` / `\n` → single space     | `"a  b\n"` → `"a b"`       |
| 3    | Repeated punctuation → single occurrence         | `"!!!"`  → `"!"`           |
| 4    | Strip (start/end)                                | `"  text  "` → `"text"`    |

---

### Pipeline Summary

```
item (dict)
  │
  ├─ title  ──┐
  ├─ text   ──┤  to_string()  →  build_content()  →  [clean_text()]  →  item["content"]
  └─ tags   ──┘
```

> **Quality Note**: 89/216,041 documents (0.04%) contain `\ufffd` in the raw
> `text` field (encoding corruption in the source Kaggle data).  
> `to_string()` sanitizes them → `content` contains **no** `\ufffd`.

## Field Description

Main fields used in this notebook (documents / queries):

- `id`: unique identifier used for indexing/evaluation.
- `title`: short title (may be missing / empty).
- `text`: main text body (may contain noisy characters in the raw data).
- `tags`: list of tags (or sometimes a string / missing value depending on source formatting).
- `category`: category label used for EDA plots and stratified inspection.
- `content`: merged field created in this notebook (`title + text + tags`) for retrieval models.

**Ground Truth (`qgts_train.json`)**

- Maps each `query_id` to relevance information.
- `relevant_doc_ids` contains the relevant document IDs (with relevance metadata in the raw file).
- Used to compute descriptive statistics (e.g., number of relevant documents per query).


## Data Loading

In [ ]:
sys.path.insert(0, str(ROOT_DIR))  # Add the project root to the path for imports.
from src.data.load import load_all
from src.data.preprocess import add_content_field


In [ ]:
# Load the datasets from the raw data directory
documents, train_queries, test_queries, gts = load_all(DATA_DIR)

print("Datasets loaded successfully:")
print(f"Number of documents: {len(documents):,}")
print(f"Number of train queries: {len(train_queries):,}")
print(f"Number of test queries: {len(test_queries):,}")
print(f"Number of ground truth entries: {len(gts):,}")

## Statistics
### General Numbers

In [ ]:
stats = {
    'Total documents': len(documents),
    'Training queries': len(train_queries),
    'Test queries': len(test_queries),
    'Total queries': len(train_queries) + len(test_queries)
}

print("\n" + "="*60)
print("GENERAL STATISTICS")
print("="*60)
for key, value in stats.items():
    print(f"{key}: {value:,}")

### Text Lengths

In [ ]:
print("\n" + "="*60)
print("LENGTH STATISTICS")
print("="*60)

# Add content fields (without cleaning for precise stats)
docs_with_content, queries_with_content = add_content_field(
    documents, train_queries, clean=False
)

# Document lengths
doc_lengths = [len(doc['content'].split()) for doc in docs_with_content]
print(f"\nDocuments:")
print(f"  - Average length: {np.mean(doc_lengths):.2f} words")
print(f"  - Median length: {np.median(doc_lengths):.2f} words")
print(f"  - Min: {np.min(doc_lengths)} words")
print(f"  - Max: {np.max(doc_lengths)} words")
print(f"  - Standard deviation: {np.std(doc_lengths):.2f} words")

# Query lengths
query_lengths = [len(q['content'].split()) for q in queries_with_content]
print(f"\nQueries:")
print(f"  - Average length: {np.mean(query_lengths):.2f} words")
print(f"  - Median length: {np.median(query_lengths):.2f} words")
print(f"  - Min: {np.min(query_lengths)} words")
print(f"  - Max: {np.max(query_lengths)} words")
print(f"  - Standard deviation: {np.std(query_lengths):.2f} words")

### Category Distribution

In [ ]:
print("\n" + "="*60)
print("CATEGORY DISTRIBUTION")
print("="*60)

# Document categories
doc_categories = Counter([doc.get('category', 'unknown') for doc in documents])
print(f"\nDocument categories (top 10):")
for cat, count in doc_categories.most_common(10):
    print(f"  - {cat}: {count:,} ({count/len(documents)*100:.2f}%)")

# Query categories
query_categories = Counter([q.get('category', 'unknown') for q in train_queries])
print(f"\nTraining query categories:")
for cat, count in query_categories.most_common(10):
    print(f"  - {cat}: {count} ({count/len(train_queries)*100:.2f}%)")

### Relevant Documents per Query

In [ ]:
print("\n" + "="*60)
print("RELEVANT DOCUMENTS PER QUERY")
print("="*60)

relevant_counts = []
for qid, gt_data in gts.items():
    # Handle ground truth format
    if isinstance(gt_data, dict):
        n_relevant = gt_data.get('total_relevant_docs', 0)
    else:
        n_relevant = len(gt_data) if isinstance(gt_data, list) else 0
    relevant_counts.append(n_relevant)

print(f"\nRelevant documents per query:")
print(f"  - Average: {np.mean(relevant_counts):.2f}")
print(f"  - Median: {np.median(relevant_counts):.2f}")
print(f"  - Min: {np.min(relevant_counts)}")
print(f"  - Max: {np.max(relevant_counts)}")
print(f"  - Total: {np.sum(relevant_counts):,} relevant query-doc pairs")

## Visualizations

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Distribution of document lengths
axes[0, 0].hist(doc_lengths, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Number of words')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of document lengths')
axes[0, 0].axvline(np.mean(doc_lengths), color='red', linestyle='--', 
                   label=f'Average: {np.mean(doc_lengths):.1f}')
axes[0, 0].legend()

# Plot 2: Distribution of query lengths
axes[0, 1].hist(query_lengths, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Number of words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of query lengths')
axes[0, 1].axvline(np.mean(query_lengths), color='red', linestyle='--', 
                   label=f'Average: {np.mean(query_lengths):.1f}')
axes[0, 1].legend()

# Plot 3: Top 10 document categories
top_categories = doc_categories.most_common(10)
cats, counts = zip(*top_categories)
axes[1, 0].barh(range(len(cats)), counts)
axes[1, 0].set_yticks(range(len(cats)))
axes[1, 0].set_yticklabels(cats)
axes[1, 0].set_xlabel('Number of documents')
axes[1, 0].set_title('Top 10 document categories')
axes[1, 0].invert_yaxis()

# Plot 4: Distribution of relevant documents per query
axes[1, 1].hist(relevant_counts, bins=range(min(relevant_counts), max(relevant_counts)+2), 
                edgecolor='black', alpha=0.7, color='orange')
axes[1, 1].set_xlabel('Number of relevant documents')
axes[1, 1].set_ylabel('Number of queries')
axes[1, 1].set_title('Relevant documents per query')
axes[1, 1].axvline(np.mean(relevant_counts), color='red', linestyle='--', 
                   label=f'Average: {np.mean(relevant_counts):.1f}')
axes[1, 1].legend()

plt.tight_layout()
PLOTS_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(PLOTS_PATH, dpi=300, bbox_inches='tight')
plt.show()

## Conclusion

In [ ]:
conclusions = """
## Exploratory Analysis Conclusions

### 1. Dataset Size
- The corpus contains {n_docs:,} documents and {n_queries} training queries
- Documents/queries ratio: {ratio:.1f}:1, indicating a large-scale retrieval problem

### 2. Textual Characteristics
- **Documents**: average length of {avg_doc:.1f} words (median: {med_doc:.1f})
  - Significant variance (standard deviation: {std_doc:.1f}), suggesting varied content types
- **Queries**: average length of {avg_query:.1f} words (median: {med_query:.1f})
  - Queries are significantly shorter than documents

### 3. Category Distribution
- {n_categories} distinct document categories
- Dominant category: {top_cat} ({top_cat_pct:.1f}% of documents)
- Unbalanced distribution → possibility to use category for reranking

### 4. Ground Truth
- On average {avg_relevant:.2f} relevant documents per query
- Maximum of {max_relevant} relevant documents for one query
- Total of {total_relevant:,} relevant query-document pairs

### 5. Implications for Retrieval
- Length variability favors BM25 (length normalization) over TF-IDF
- Short queries require robust techniques (semantic embeddings)
- Categorical distribution can serve as a signal for reranking (Phase 2)
"""

print("\n" + "="*60)
print(conclusions.format(
    n_docs=len(documents),
    n_queries=len(train_queries),
    ratio=len(documents)/len(train_queries),
    avg_doc=np.mean(doc_lengths),
    med_doc=np.median(doc_lengths),
    std_doc=np.std(doc_lengths),
    avg_query=np.mean(query_lengths),
    med_query=np.median(query_lengths),
    n_categories=len(doc_categories),
    top_cat=doc_categories.most_common(1)[0][0],
    top_cat_pct=doc_categories.most_common(1)[0][1]/len(documents)*100,
    avg_relevant=np.mean(relevant_counts),
    max_relevant=np.max(relevant_counts),
    total_relevant=np.sum(relevant_counts)
))
print("="*60)

## Saving

What this notebook outputs: `docs_with_content.json`, `queries_train_with_content.json`, `queries_test_with_content.json`, and `eda_plots.png`.


In [ ]:
# Add content with cleaning for models
docs_clean, queries_train_clean = add_content_field(documents, train_queries, clean=True)
_, queries_test_clean = add_content_field([], test_queries, clean=True)

# Create processed folder if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save
with open(PROCESSED_DIR / "docs_with_content.json", "w", encoding="utf-8") as f:
    json.dump(docs_clean, f, ensure_ascii=False, indent=2)

with open(PROCESSED_DIR / "queries_train_with_content.json", "w", encoding="utf-8") as f:
    json.dump(queries_train_clean, f, ensure_ascii=False, indent=2)

with open(PROCESSED_DIR / "queries_test_with_content.json", "w", encoding="utf-8") as f:
    json.dump(queries_test_clean, f, ensure_ascii=False, indent=2)

print("\nEnriched data saved in data/processed/")

## Quality Check

In [ ]:
# --- Inspection of docs containing \ufffd ---
bad_docs = [d for d in docs_with_content if isinstance(d.get("content"), str) and "\ufffd" in d["content"]]
print(f"Docs with \\ufffd : {len(bad_docs)}")

# Show some examples (id + where the problem is)
for d in bad_docs[:10]:
    doc_id = d.get("id", "<no-id>")
    title = d.get("title", "")
    text  = d.get("text", "")
    tags  = d.get("tags", [])
    print("\n---", doc_id, "---")
    print("in title?", isinstance(title, str) and "\ufffd" in title)
    print("in text? ", isinstance(text, str) and "\ufffd" in text)
    if isinstance(tags, list):
        print("in tags? ", any(isinstance(t, str) and "\ufffd" in t for t in tags))
    else:
        print("in tags? ", isinstance(tags, str) and "\ufffd" in tags)

In [ ]:
# ── Quality check for the `content` field ──────────────────────────────────
# `content` is produced by build_content() → to_string() which already
# replaces '\ufffd' with ' '. The 89 docs containing \ufffd in `text`
# therefore do not pollute `content`. The assertion below should pass 0.

_, queries_test_with_content = add_content_field(documents, test_queries, clean=False)

collections = {
    "documents": docs_with_content,           # 216 041 items
    "queries (train)": queries_with_content,   # 327 items
    "queries (test)": queries_test_with_content, # 141 items
}

print("=" * 60)
print("QUALITY CHECK — `content` field")
print("=" * 60)

grand_total = 0

for name, items in collections.items():
    n = len(items)
    missing_key = sum(1 for d in items if "content" not in d)
    is_none = sum(1 for d in items if d.get("content") is None)
    is_empty = sum(1 for d in items
                    if isinstance(d.get("content"), str)
                    and d["content"].strip() == "")
    too_short = sum(1 for d in items
                    if isinstance(d.get("content"), str)
                    and 0 < len(d["content"].split()) < 3)
    bad_enc = sum(1 for d in items
                  if isinstance(d.get("content"), str)
                  and "\ufffd" in d["content"])
    contents = [d["content"] for d in items
                if isinstance(d.get("content"), str)]
    duplicates = n - len(set(contents))

    subtotal = missing_key + is_none + is_empty + too_short + bad_enc
    grand_total += subtotal

    print(f"\n▸ {name} ({n:,} items)")
    print(f"   Missing 'content' key : {missing_key}")
    print(f"   None value            : {is_none}")
    print(f"   Empty/whitespace      : {is_empty}")
    print(f"   Too short (< 3 words) : {too_short}")
    print(f"   U+FFFD character      : {bad_enc}")
    print(f"   [info] duplicates     : {duplicates}")
    print(f"   {'─' * 42}")
    print(f"   Critical anomalies    : {subtotal}")

print("\n" + "=" * 60)
print(f"Overall result         : {grand_total} critical anomaly(ies).")
print("=" * 60 + "\n")

# ── Justification if > 0 (should not appear) ───────────────────────────────
if grand_total > 0:
    print(
        f"{grand_total} anomaly(ies) detected — JUSTIFICATION:\n"
        "   • Origin   : U+FFFD character (encoding) present in raw\n"
        "               Kaggle data (field `text`).\n"
        "   • Scope    : 89 / 216 041 documents (0.04 % of the corpus).\n"
        "   • Cause    : to_string() was not applied in this pipeline.\n"
        "   • Decision : fix build_content() or to_string() in\n"
        "               src/data/preprocess.py before Phase 2 (embeddings).\n"
    )

# ── Blocking assertion ────────────────────────────────────────────────────
assert grand_total == 0, (
    f"{grand_total} anomalies in `content` — "
    "fix build_content() / to_string() in src/data/preprocess.py"
)
print("Quality of `content` field: OK — 0 critical anomalies")

## Summary for Report

In [ ]:
# ── Numeric summary for the report ────────────────────────────────────────
report = {
    # Volumes
    "Number of documents"                  : len(documents),
    "Number of training queries"          : len(train_queries),
    "Number of test queries"              : len(test_queries),
    "Docs / train queries ratio"          : round(len(documents) / len(train_queries), 1),

    # Lengths (field `content`, no cleaning)
    "Avg. document length (words)"        : round(np.mean(doc_lengths), 1),
    "Med. document length (words)"        : round(np.median(doc_lengths), 1),
    "Std dev document length (words)"     : round(np.std(doc_lengths), 1),
    "Avg. query length (words)"           : round(np.mean(query_lengths), 1),
    "Med. query length (words)"           : round(np.median(query_lengths), 1),

    # Categories
    "Distinct categories count"           : len(doc_categories),
    "Dominant doc category"               : doc_categories.most_common(1)[0][0],
    "Dominant category share (%)"         : round(
                                               doc_categories.most_common(1)[0][1]
                                               / len(documents) * 100, 1),

    # Ground truth
    "Avg relevant docs per query"         : round(np.mean(relevant_counts), 2),
    "Max relevant docs (one query)"       : int(np.max(relevant_counts)),
    "Total relevant query-doc pairs"      : int(np.sum(relevant_counts)),
}

print("=" * 60)
print("  EDA SUMMARY — KEY NUMBERS FOR REPORT")
print("=" * 60)
for k, v in report.items():
    print(f"  {k:<45} {v:>10}")
print("=" * 60)